In [0]:
#************************************************************************************************************************************
#*                                                                                                                                  *
#*   NOTEBOOK:     Elig_PartE_CompareTwoVETables.                                                                                   *
#*                                                                                                                                  *
#*   DESCRIPTION:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT PARMS:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT FILES:                                                                                                                   *

#*                                                                                                                                  *
#*                                                                                                                                  *
#*   OUTPUT FILE:                                                                                                                   *
#*                                                                                                                                  *
#*   EXITS:       0 - success                                                                                                       *
#*                <> 0 - failure                                                                                                    *
#*                                                                                                                                  *
#************************************************************************************************************************************
#*                                                                                                                                  *
#*                                                 Modification Log                                                                 *
#*                                                                                                                                  *
#*    Date     CO                 Author              Description                                                                   *
#* ---------- ------------------  -----------------   ------------------------------------------------------------------------------*
#* 11/12/2025 CCRB70930/CO#43342  Jaime Zavala        Initial Release.                                                              *
#* 11/13/2025 CCRB70930/CO#43342  Jaime Zavala        Added Reconciliation Count Logic.                                             *
#* 11/17/2025 CCRB70930/CO#43342  Jaime Zavala        Modified to met QA's Spreadsheet.                                             *
#************************************************************************************************************************************

In [0]:
#-----------
# DBX Parms
#-----------
dbutils.widgets.text('catalog', 'oh_apm_stg')  
dbutils.widgets.text('schema_name', 'etl_qa')  

#-------------
#  Tables
#-------------

dbutils.widgets.text('DiffAnlysTblNm', 'EligVE_Part_E_DiffAnlys')
dbutils.widgets.text('PrevTblNm', 'EDW_VEN116FA_PartE_Previous')
dbutils.widgets.text('CurntTblNm', 'EDW_VEN116FA_PartE_Current')

#-------------------
# Getters Section
#-------------------
catalog     = dbutils.widgets.get('catalog')
schema_name = dbutils.widgets.get('schema_name')
DiffAnlysTblNm = dbutils.widgets.get('DiffAnlysTblNm')
CurntTblNm = dbutils.widgets.get('CurntTblNm')
PrevTblNm = dbutils.widgets.get('PrevTblNm')

#---------------
# Print Section
#---------------
print("catalog:", catalog)
print("schema:", schema_name)
print("Diff Analysis Table Name:", DiffAnlysTblNm)
print("Current Table Name:", CurntTblNm)
print("Previous Table Name:", PrevTblNm)


In [0]:

%sql
create or replace table ${catalog}.${schema_name}.${DiffAnlysTblNm} AS 
SELECT 
    -- Common identifiers
     COALESCE(PrevTbl.ID_MEDICAID, CurntTbl.ID_MEDICAID) AS ID_MEDICAID       -- This is the common identifier
    ,COALESCE(PrevTbl.DTE_EFFECTIVE, CurntTbl.DTE_EFFECTIVE) AS DTE_EFFECTIVE -- This is the common identifier
	-- None Compare Fileds
    ,CASE WHEN PrevTbl.REPORT_DTE              != CurntTbl.REPORT_DTE                THEN PrevTbl.REPORT_DTE              ELSE NULL END AS PrevTbl_REPORT_DTE        -- Derived1
    ,CASE WHEN PrevTbl.REPORT_DTE              != CurntTbl.REPORT_DTE                THEN CurntTbl.REPORT_DTE             ELSE NULL END AS CurntTbl_REPORT_DTE
    -- Compare Fields
    ,CASE WHEN PrevTbl.ENRL_SPAN_TYP           != CurntTbl.ENRL_SPAN_TYP             THEN PrevTbl.ENRL_SPAN_TYP           ELSE NULL END AS PrevTbl_ENRL_SPAN_TYP     -- Derived2
    ,CASE WHEN PrevTbl.ENRL_SPAN_TYP           != CurntTbl.ENRL_SPAN_TYP             THEN CurntTbl.ENRL_SPAN_TYP          ELSE NULL END AS CurntTbl_ENRL_SPAN_TYP
    ,CASE WHEN PrevTbl.NUM_CASE                != CurntTbl.NUM_CASE                  THEN PrevTbl.NUM_CASE                ELSE NULL END AS PrevTbl_NUM_CASE
    ,CASE WHEN PrevTbl.NUM_CASE                != CurntTbl.NUM_CASE                  THEN CurntTbl.NUM_CASE               ELSE NULL END AS CurntTbl_NUM_CASE
    ,CASE WHEN PrevTbl.DTE_END                 != CurntTbl.DTE_END                   THEN PrevTbl.DTE_END                 ELSE NULL END AS PrevTbl_DTE_END
    ,CASE WHEN PrevTbl.DTE_END                 != CurntTbl.DTE_END                   THEN CurntTbl.DTE_END                ELSE NULL END AS CurntTbl_DTE_END
    ,CASE WHEN PrevTbl.ID_PROVIDER_MCAID       != CurntTbl.ID_PROVIDER_MCAID         THEN PrevTbl.ID_PROVIDER_MCAID       ELSE NULL END AS PrevTbl_ID_PROVIDER_MCAID
    ,CASE WHEN PrevTbl.ID_PROVIDER_MCAID       != CurntTbl.ID_PROVIDER_MCAID         THEN CurntTbl.ID_PROVIDER_MCAID      ELSE NULL END AS CurntTbl_ID_PROVIDER_MCAID
           FROM ${catalog}.${schema_name}.${CurntTblNm} CurntTbl
FULL OUTER JOIN ${catalog}.${schema_name}.${PrevTblNm} PrevTbl ON PrevTbl.ID_MEDICAID    = CurntTbl.ID_MEDICAID 
                                                              AND PrevTbl.DTE_EFFECTIVE = CurntTbl.DTE_EFFECTIVE
WHERE 
       PrevTbl.NUM_CASE                != CurntTbl.NUM_CASE                  
    OR PrevTbl.ENRL_SPAN_TYP           != CurntTbl.ENRL_SPAN_TYP                     
    OR PrevTbl.DTE_END                 != CurntTbl.DTE_END                
    OR PrevTbl.ID_PROVIDER_MCAID       != CurntTbl.ID_PROVIDER_MCAID
   AND PrevTbl.DTE_EFFECTIVE is not null
;

In [0]:
%sql
 select count(*) from ${catalog}.${schema_name}.${DiffAnlysTblNm};

In [0]:
%sql
 select * from ${catalog}.${schema_name}.${DiffAnlysTblNm}
 ;

In [0]:
%sql
WITH
  Sum_Total AS (
    SELECT count(*) cntTotal
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE TRUE 
  ),
  Cnt_ENRL_SPAN_TYP AS (
    SELECT COUNT(*) AS Diff_ENRL_SPAN_TYP
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE CurntTbl_ENRL_SPAN_TYP IS NOT NULL
       OR PrevTbl_ENRL_SPAN_TYP IS NOT NULL
  ),
  Cnt_NUM_CASE AS (
    SELECT COUNT(*) AS Diff_NUM_CASE
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE CurntTbl_NUM_CASE IS NOT NULL
       OR PrevTbl_NUM_CASE IS NOT NULL
  ),
  Cnt_DTE_END AS (
    SELECT COUNT(*) AS Diff_DTE_END
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE CurntTbl_DTE_END IS NOT NULL
       OR PrevTbl_DTE_END IS NOT NULL
  ),
  Cnt_ID_PROVIDER_MCAID AS (
    SELECT COUNT(*) AS Diff_ID_PROVIDER_MCAID
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE CurntTbl_ID_PROVIDER_MCAID IS NOT NULL
       OR PrevTbl_ID_PROVIDER_MCAID IS NOT NULL
  )

SELECT
  format_number(t1.Diff_ENRL_SPAN_TYP, 0)                        AS Diff_ENRL_SPAN_TYP,
  format_number(CASE WHEN t5.cntTotal = 0 THEN 0
                  ELSE t1.Diff_ENRL_SPAN_TYP * 100 / t5.cntTotal
             END, '###.#')                                  AS Pcntg_ENRL_SPAN_TYP,
  format_number(t2.Diff_NUM_CASE, 0)                        AS Diff_NUM_CASE,
  format_number(CASE WHEN t5.cntTotal = 0 THEN 0
                  ELSE t2.Diff_NUM_CASE * 100 / t5.cntTotal
             END, '###.#')                                  AS Pcntg_NUM_CASE,
  format_number(t3.Diff_DTE_END, 0)                       AS Diff_DTE_END,
  format_number(CASE WHEN t5.cntTotal = 0 THEN 0
                  ELSE t3.Diff_DTE_END * 100 / t5.cntTotal
             END, '###.#')                                  AS Pcntg_DTE_END,
  format_number(t4.Diff_ID_PROVIDER_MCAID, 0)                AS Diff_ID_PROVIDER_MCAID,
  format_number(CASE WHEN t5.cntTotal = 0 THEN 0
                  ELSE t4.Diff_ID_PROVIDER_MCAID * 100 / t5.cntTotal
             END, '###.#')                                  AS Pcntg_ID_PROVIDER_MCAID,
  format_number(t5.cntTotal, 0)                             AS cntTotal
FROM
  Cnt_ENRL_SPAN_TYP t1,
  Cnt_NUM_CASE t2,
  Cnt_DTE_END t3,
  Cnt_ID_PROVIDER_MCAID t4,
  Sum_Total t5
;

In [0]:
sql_out = spark.sql(f"""
WITH
  Match_Counts AS (
    SELECT DISTINCT PrevTbl.ID_MEDICAID
                   ,PrevTbl.DTE_EFFECTIVE 
          FROM {catalog}.{schema_name}.{CurntTblNm} CurntTbl
          JOIN {catalog}.{schema_name}.{PrevTblNm} PrevTbl ON PrevTbl.ID_MEDICAID    = CurntTbl.ID_MEDICAID 
                                                             AND PrevTbl.DTE_EFFECTIVE  = CurntTbl.DTE_EFFECTIVE
    WHERE TRUE  
      AND PrevTbl.DTE_EFFECTIVE is not null
  ),
  Get_CntsNew_VsOld AS (
    SELECT COUNT(*) AS CntNew_VsOld
    FROM {catalog}.{schema_name}.{CurntTblNm} CurntTbl
    WHERE TRUE
      AND CurntTbl.DTE_EFFECTIVE is not null
      AND NOT EXISTS
       (
        SELECT 1
          FROM Match_Counts t1
         WHERE TRUE 
           AND CurntTbl.ID_MEDICAID    = t1.ID_MEDICAID
           AND CurntTbl.DTE_EFFECTIVE = t1.DTE_EFFECTIVE
       )
  ),
  Get_CntsOld_VsNew AS (
    SELECT COUNT(*) AS CntOld_VsNew
    FROM {catalog}.{schema_name}.{PrevTblNm} PrevTbl
    WHERE TRUE
       AND PrevTbl.DTE_EFFECTIVE is not null
      AND NOT EXISTS
       (
        SELECT 1
          FROM Match_Counts t4
         WHERE TRUE 
           AND PrevTbl.ID_MEDICAID    = t4.ID_MEDICAID
           AND PrevTbl.DTE_EFFECTIVE = t4.DTE_EFFECTIVE
       )
  )
SELECT DISTINCT format_number((SELECT COUNT(*) FROM Match_Counts), 0) AS TotalMatch
     , format_number(t2.CntNew_VsOld, 0) AS RecordsNewVsOld
     , format_number(t3.CntOld_VsNew, 0) AS RecordsOldVsNew
  FROM Match_Counts t1
     , Get_CntsNew_VsOld t2
     , Get_CntsOld_VsNew t3
;
               """)
display(sql_out)

Databricks data profile. Run in Databricks to view.

In [0]:
sql_out = spark.sql(f"""
WITH
  Recip_Counts AS (
    SELECT COUNT( DISTINCT ID_MEDICAID) AS RecipCnt
          FROM {catalog}.{schema_name}.{CurntTblNm}
    WHERE TRUE
  ),
  EligRecip_Counts AS (
    SELECT COUNT(*)   AS ProvMcaidCnt2
         , SUM(IDCnt) AS McaidCnt2
    FROM 
    (SELECT DISTINCT ID_PROVIDER_MCAID
          , COUNT(DISTINCT ID_MEDICAID) AS IDCnt
       FROM {catalog}.{schema_name}.{CurntTblNm}
      WHERE TRUE
        AND current_date() BETWEEN DTE_EFFECTIVE AND DTE_END
      GROUP BY 1
    )
  ),
  ProvMcaid_Counts AS (
    SELECT COUNT(*)   AS ProvMcaidCnt
         , SUM(IDCnt) AS McaidCnt
    FROM 
    (SELECT DISTINCT ID_PROVIDER_MCAID
          , COUNT(DISTINCT ID_MEDICAID) AS IDCnt
       FROM {catalog}.{schema_name}.{CurntTblNm}
      GROUP BY 1
    )
  )
--
-- Main SQL
--
SELECT format_number(t1.RecipCnt, 0) AS TotalRecip
     , format_number(t2.ProvMcaidCnt, 0) AS TotalProvMcaid
     , format_number(t2.McaidCnt, 0) AS TotalMcaid
     , format_number(t3.ProvMcaidCnt2, 0) AS TotalEligProvMcaid
     , format_number(t3.McaidCnt2, 0) AS TotalEligMcaid
  FROM Recip_Counts t1
     , ProvMcaid_Counts t2
     , EligRecip_Counts t3
;
               """)
display(sql_out)